# Baseline: TrafficGPT (adapted)

A compact decoder-only transformer in the spirit of TrafficGPT (Qu et al., 2024). The original model works on byte-level tokens of whole packets with linear attention; here every packet becomes two tokens, one for its payload and one for its IAT, each quantized into 50 bins that are uniform in log(1 + x). A flow is the token sequence [SOS, payload_1, IAT_1, payload_2, IAT_2, ..., EOS] of at most 512 tokens; generation samples with top-k (k = 10) until EOS, and flows are unconditional, so their length is learned from the EOS token.

Run by `scripts/run_matrix.py` after the proposed model of the same dataset and seed, whose `split.json` and `test.csv` it reuses. The packets are cleaned with the same function as for the proposed model, and the run stops if its test flows differ from `test.csv`.

In [ ]:
SEED         = 0
DATASET      = 'facebook_video'
RAW_PATH_STR = 'data/facebook_video.csv'
OUT_DIR      = 'runs/facebook_video/seed0'
MODEL_KEY    = 'trafficgpt'
TEST_SIZE    = 0.3              # not used: the split is read from split.json

In [ ]:
import os
import random
import sys
import time
import warnings
from pathlib import Path

import dill
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import KBinsDiscretizer
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, load_split_checked, postprocess_synthetic

ART = Path(OUT_DIR) / 'artifacts' / MODEL_KEY
ART.mkdir(parents=True, exist_ok=True)
torch.use_deterministic_algorithms(True)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMBED_DIM      = 128
NUM_HEADS      = 4
NUM_LAYERS     = 4
DROPOUT        = 0.1
MAX_SEQ_LEN    = 512     # tokens per flow
BATCH_SIZE     = 1
EPOCHS         = 20
LEARNING_RATE  = 1e-3
TOP_K          = 10      # top-k sampling
TEMPERATURE    = 1.0
N_BINS_PAYLOAD = 50
N_BINS_TIME    = 50

In [ ]:
df = load_packets(RAW_PATH_STR)
train_flows, test_flows = load_split_checked(df, OUT_DIR)


class MetadataTokenizer:
    # token ids: 0 PAD, 1 SOS, 2 EOS, then N_BINS_PAYLOAD payload tokens and N_BINS_TIME IAT tokens;
    # the bins are uniform in log(1 + x) between the minimum and maximum of the training packets
    def __init__(self, n_bins_payload=N_BINS_PAYLOAD, n_bins_time=N_BINS_TIME):
        self.n_p = n_bins_payload
        self.n_t = n_bins_time
        self.est_payload = KBinsDiscretizer(n_bins=self.n_p, encode='ordinal', strategy='uniform', random_state=SEED)
        self.est_time = KBinsDiscretizer(n_bins=self.n_t, encode='ordinal', strategy='uniform', random_state=SEED)
        self.special_tokens = {'PAD': 0, 'SOS': 1, 'EOS': 2}
        self.offset_payload = 3
        self.offset_time = 3 + self.n_p
        self.vocab_size = 3 + self.n_p + self.n_t

    def fit(self, df_train):
        p_data = np.log1p(df_train['payload_length'].values).reshape(-1, 1)
        t_data = np.log1p(df_train['time_diff'].values).reshape(-1, 1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            self.est_payload.fit(p_data)
            self.est_time.fit(t_data)
        print(f'tokenizer fitted, vocabulary size {self.vocab_size}')

    def tokenize(self, payload_val, time_val):
        # values outside the fitted range are clipped to it
        p_log = np.clip(np.log1p(payload_val), self.est_payload.bin_edges_[0][0], self.est_payload.bin_edges_[0][-1])
        t_log = np.clip(np.log1p(time_val), self.est_time.bin_edges_[0][0], self.est_time.bin_edges_[0][-1])
        p_bin = self.est_payload.transform(np.array([[p_log]]).reshape(-1, 1)).astype(int).item()
        t_bin = self.est_time.transform(np.array([[t_log]]).reshape(-1, 1)).astype(int).item()
        return p_bin + self.offset_payload, t_bin + self.offset_time

    def detokenize(self, token_id):
        # a payload or IAT token is mapped to the center of its bin
        if token_id < 3:
            return None, 'SPECIAL'
        if self.offset_payload <= token_id < self.offset_time:
            bin_idx = max(0, min(token_id - self.offset_payload, self.n_p - 1))
            edges = self.est_payload.bin_edges_[0]
            return np.expm1((edges[bin_idx] + edges[bin_idx + 1]) / 2.0), 'PAYLOAD'
        elif token_id >= self.offset_time:
            bin_idx = max(0, min(token_id - self.offset_time, self.n_t - 1))
            edges = self.est_time.bin_edges_[0]
            return np.expm1((edges[bin_idx] + edges[bin_idx + 1]) / 2.0), 'TIME'
        return 0.0, 'UNKNOWN'


train_df_raw = df[df['flow_id'].isin(train_flows)]
tokenizer = MetadataTokenizer()
tokenizer.fit(train_df_raw)


class TrafficGPTDataset(Dataset):
    # one token sequence [SOS, p1, t1, p2, t2, ..., EOS] per flow, padded to max_len
    def __init__(self, frame, tokenizer, max_len=MAX_SEQ_LEN):
        self.samples = []
        for _, group in tqdm(frame.groupby('flow_id'), desc='tokenizing'):
            token_seq = [tokenizer.special_tokens['SOS']]
            for p, t in zip(group['payload_length'].values, group['time_diff'].values):
                token_seq.extend(tokenizer.tokenize(p, t))
                if len(token_seq) >= max_len - 1:
                    break
            token_seq.append(tokenizer.special_tokens['EOS'])
            if len(token_seq) < max_len:
                token_seq += [tokenizer.special_tokens['PAD']] * (max_len - len(token_seq))
            self.samples.append(torch.tensor(token_seq, dtype=torch.long))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq = self.samples[idx]
        return seq[:-1], seq[1:]                    # next-token prediction


train_ds = TrafficGPTDataset(train_df_raw, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f'{len(train_ds)} training sequences')

In [ ]:
class TrafficGPT(nn.Module):
    # decoder-only transformer: token and position embeddings, pre-norm self-attention layers with a
    # causal mask, and a linear head over the vocabulary
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, dropout, max_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_len, embed_dim)
        decoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.register_buffer('causal_mask', torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1))

    def forward(self, idx):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        x = self.transformer_decoder(x, mask=self.causal_mask[:t, :t], is_causal=True)
        return self.head(self.ln_f(x))


model = TrafficGPT(vocab_size=tokenizer.vocab_size, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
                   dropout=DROPOUT, max_len=MAX_SEQ_LEN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.special_tokens['PAD'])
model.train()
for epoch in range(EPOCHS):
    total_loss, start_t = 0, time.time()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch {epoch + 1}/{EPOCHS}: loss {total_loss / len(train_loader):.4f} ({time.time() - start_t:.1f} s)')
torch.save(model.state_dict(), ART / 'trafficgpt_weights.pth')

In [ ]:
def generate_flow(model, tokenizer, max_len=MAX_SEQ_LEN, top_k=TOP_K, temperature=TEMPERATURE):
    # sample tokens until EOS or max_len, then keep the (payload, IAT) pairs
    model.eval()
    ctx = torch.tensor([[tokenizer.special_tokens['SOS']]], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_len):
            logits = model(ctx)
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
            next_token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            if next_token.item() == tokenizer.special_tokens['EOS']:
                break
            ctx = torch.cat((ctx, next_token), dim=1)
            if ctx.size(1) >= max_len:
                break
    tokens = ctx[0].cpu().numpy()[1:]
    vals_p, vals_t, i = [], [], 0
    while i < len(tokens) - 1:
        val1, type1 = tokenizer.detokenize(tokens[i])
        val2, type2 = tokenizer.detokenize(tokens[i + 1])
        if type1 == 'PAYLOAD' and type2 == 'TIME':
            vals_p.append(val1)
            vals_t.append(val2)
            i += 2
        else:                                       # misaligned token: skipped
            i += 1
    return vals_p, vals_t


records = []
for fid in tqdm(test_flows, desc='flows'):          # one unconditional flow per test flow
    p_seq, t_seq = generate_flow(model, tokenizer, top_k=TOP_K, temperature=TEMPERATURE)
    for p, t in zip(p_seq, t_seq):
        records.append([fid, round(max(0, min(p, 1500))), float(max(1e-6, t))])
synth_df = pd.DataFrame(records, columns=['flow_id', 'payload_length', 'time_diff'])

with open(ART / 'trafficgpt_tokenizer.pkl', 'wb') as fh:
    dill.dump(tokenizer, fh)

synth_df = postprocess_synthetic(synth_df)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'saved {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')